# - Live Weather Prediction

This notebook fetches real-time weather data from OpenWeather API, preprocesses features to align with the trained machine learning pipeline, and performs live temperature predictions using the trained Random Forest model.

In [1]:
import requests
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime

# Project Root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Load Model and Encoders
model = joblib.load(ROOT / "models" / "weather_model.pkl")
le_city = joblib.load(ROOT / "models" / "le_city.pkl")
le_weather = joblib.load(ROOT / "models" / "le_weather.pkl")
le_day = joblib.load(ROOT / "models" / "le_day.pkl")
le_season = joblib.load(ROOT / "models" / "le_season.pkl")

print("✅ Model & Encoders loaded successfully!")

✅ Model & Encoders loaded successfully!


In [2]:
print("City Encoders (sample):", list(le_city.classes_[:10]))
print("Weather Classes:", list(le_weather.classes_))
print("Day Classes:", list(le_day.classes_))
print("Season Classes:", list(le_season.classes_))

City Encoders (sample): ['Adilabad', 'Ahmedabad', 'Ajmer', 'Amravati', 'Anand', 'Aurangabad', 'Barmer', 'Bengaluru', 'Bharuch', 'Bhavnagar']
Weather Classes: ['Clear', 'Cloudy', 'Rain', 'Thunderstorm']
Day Classes: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
Season Classes: ['Monsoon', 'Post-Monsoon', 'Summer', 'Winter']


In [3]:
def get_season(month: int) -> str:
    """Map month integer to season matching model training dataset."""
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

In [4]:
API_KEY = "1becb0fb56249486d02f3d4f89203315"

def fetch_live_weather(city: str, api_key: str = API_KEY):
    """Fetch live weather from OpenWeather API."""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }
    try:
        response = requests.get(url, params=params)
        data = response.json()
        if response.status_code != 200:
            print(f"❌ API Error: {data.get('message', 'Failed to fetch data')}")
            return None
        
        return {
            "city": city,
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"],
            "pressure": data["main"]["pressure"],
            "wind_speed": data["wind"]["speed"],
            "cloudiness": data["clouds"]["all"],
            "weather": data["weather"][0]["main"],
            "rainfall": data.get("rain", {}).get("1h", 0.0),
            "timestamp": datetime.now()
        }
    except Exception as e:
        print(f"❌ Network Error: {e}")
        return None

In [5]:
def prepare_prediction_data(weather):
    """Transform live weather dictionary into model feature dataframe."""
    if weather is None:
        return None
    
    try:
        city_name = weather["city"]
        matched_city = next((c for c in le_city.classes_ if c.lower() == city_name.lower()), city_name)
        
        if matched_city in le_city.classes_:
            city_encoded = le_city.transform([matched_city])[0]
        else:
            print(f"❌ City '{city_name}' not found in trained city encoder.")
            return None
        
        weather_main = weather["weather"]
        weather_map = {
            "Clouds": "Cloudy",
            "Drizzle": "Rain",
            "Mist": "Cloudy",
            "Fog": "Cloudy",
            "Haze": "Cloudy",
            "Smoke": "Cloudy"
        }
        normalized_weather = weather_map.get(weather_main, weather_main)
        
        if normalized_weather in le_weather.classes_:
            weather_encoded = le_weather.transform([normalized_weather])[0]
        else:
            weather_encoded = 0
        
        now = weather["timestamp"]
        day_name = now.strftime("%A")
        season_name = get_season(now.month)
        
        day_encoded = le_day.transform([day_name])[0]
        season_encoded = le_season.transform([season_name])[0]
        
        X = pd.DataFrame([{
            "City": city_encoded,
            "Humidity": weather["humidity"],
            "Pressure": weather["pressure"],
            "Wind_Speed": weather["wind_speed"],
            "Cloud_Cover": weather["cloudiness"],
            "Weather": weather_encoded,
            "Rainfall": weather["rainfall"],
            "Year": now.year,
            "Month": now.month,
            "Day": now.day,
            "Hour": now.hour,
            "DayOfWeek": day_encoded,
            "Season": season_encoded
        }])
        
        return X
    except Exception as e:
        print(f"❌ Preprocessing Error: {e}")
        return None

In [6]:
def predict_weather(city: str, api_key: str = API_KEY):
    """Predict live weather for a given city."""
    weather = fetch_live_weather(city, api_key)
    if weather is None:
        return None
    
    X = prepare_prediction_data(weather)
    if X is None:
        return None
    
    predicted_temp = model.predict(X)[0]
    
    print("\n" + "=" * 45)
    print(f"🌡️ LIVE WEATHER PREDICTION FOR {city.upper()}")
    print("=" * 45)
    print(f"📍 City:                {weather['city']}")
    print(f"🕒 Timestamp:           {weather['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"🌤️ Current Weather:      {weather['weather']}")
    print(f"🌡️ Actual Live Temp:     {weather['temperature']} °C")
    print(f"🔮 Predicted Temp (ML):  {predicted_temp:.2f} °C")
    print(f"💧 Humidity:             {weather['humidity']} %")
    print(f"🌬️ Wind Speed:           {weather['wind_speed']} m/s")
    print(f"☁️ Cloud Cover:          {weather['cloudiness']} %")
    print(f"🌧️ Rainfall (1h):        {weather['rainfall']} mm")
    print("=" * 45)
    
    return {
        "city": weather["city"],
        "actual_temp": weather["temperature"],
        "predicted_temp": predicted_temp,
        "weather": weather["weather"],
        "timestamp": weather["timestamp"]
    }

In [7]:
# Test live prediction on a city
city_input = "Mangaluru"
result = predict_weather(city_input, API_KEY)


🌡️ LIVE WEATHER PREDICTION FOR MANGALURU
📍 City:                Mangaluru
🕒 Timestamp:           2026-08-07 07:40:04
🌤️ Current Weather:      Rain
🌡️ Actual Live Temp:     25.87 °C
🔮 Predicted Temp (ML):  26.39 °C
💧 Humidity:             86 %
🌬️ Wind Speed:           6.8 m/s
☁️ Cloud Cover:          100 %
🌧️ Rainfall (1h):        0.62 mm
